# ELLA VIDEO ENGINE — Wan 2.2 TI2V 5B\n\n**Kaggle GPU notebook | Text-to-Video + Image-to-Video**\n\n**Protected fast-start edition:** the original notebook is backed up in `notebooks/backups/`. A verified `sd-cli` binary is reused without running CMake or compiling again. New builds detect the actual GPU architecture, preserve the build directory, and compile only `sd-cli`. Set **Accelerator = GPU** and **Internet = On**. Kaggle session and GPU quotas still apply.\n

In [ ]:
import os, subprocess, shutil, json, pathlib, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert shutil.which('nvidia-smi'), 'Enable a Kaggle GPU accelerator in Notebook settings.'
required = {'git': 'git', 'cmake': 'cmake', 'g++': 'build-essential', 'ffmpeg': 'ffmpeg'}
missing = [pkg for cmd, pkg in required.items() if shutil.which(cmd) is None]
if missing:
    print('Installing missing system tools:', missing, flush=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', *missing], check=True)
for module, package in [('huggingface_hub', 'huggingface_hub'), ('requests', 'requests')]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)
print('ELLA environment ready; Kaggle PyTorch/CUDA packages are reused.')

## 1. Minimal-source CUDA build (ELLA sd-cli)
This cell uses a shallow, blob-filtered sparse checkout for the engine and a separate **blob-filtered partial clone of the exact pinned GGML submodule commit**. GGML is checked out only for its CMake files, headers, CPU/base sources, and CUDA backend sources; unused GGML backends, tests, examples, docs, assets, WebUI/server, and media submodules are not checked out. The local examples CMake file is restricted to the CLI target. A verified CUDA binary/build is reused when it matches the current GPU architecture. The first CUDA compile is still required on a fresh Kaggle runtime.


In [ ]:
%cd /kaggle/working
import os, subprocess, shutil, pathlib, torch

REPO_DIR = pathlib.Path('/kaggle/working/stable-diffusion.cpp')
BUILD_DIR = REPO_DIR / 'build'
SDCLI_PATH = BUILD_DIR / 'bin' / 'sd-cli'
major, minor = torch.cuda.get_device_capability(0)
gpu_arch = f'{major}{minor}'
print('GPU:', torch.cuda.get_device_name(0), '| CUDA capability:', gpu_arch, flush=True)

def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, **kwargs)

# Keep an already-built, working CUDA CLI; do not download or rebuild it again.
cache_file = BUILD_DIR / 'CMakeCache.txt'
cache_text = cache_file.read_text(errors='ignore') if cache_file.is_file() else ''
cache_ok = ('SD_CUDA:BOOL=ON' in cache_text and
            f'CMAKE_CUDA_ARCHITECTURES:STRING={gpu_arch}' in cache_text)
binary_ok = SDCLI_PATH.is_file() and cache_ok
if binary_ok:
    try:
        p = subprocess.run([str(SDCLI_PATH), '--help'], capture_output=True,
                           text=True, timeout=30)
        binary_ok = p.returncode == 0
    except Exception:
        binary_ok = False

if not binary_ok:
    # Clone only Git metadata first; sparse checkout downloads blobs only for
    # the paths below, rather than checking out the repository's assets/docs.
    if not (REPO_DIR / '.git').is_dir():
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run(['git', 'clone', '--depth=1', '--filter=blob:none',
             '--no-checkout', '--single-branch',
             'https://github.com/leejet/stable-diffusion.cpp.git', REPO_DIR])

    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('ELLA: source checkout is not a valid Git repository.')

    run(['git', '-C', REPO_DIR, 'sparse-checkout', 'init', '--no-cone'])
    sparse_paths = [
        '/.gitmodules', '/LICENSE', '/CMakeLists.txt',
        '/cmake/', '/include/', '/src/', '/thirdparty/', '/ggml/',
        '/examples/CMakeLists.txt', '/examples/cli/', '/examples/common/'
    ]
    run(['git', '-C', REPO_DIR, 'sparse-checkout', 'set', '--no-cone', *sparse_paths])
    run(['git', '-C', REPO_DIR, 'checkout', '--force'])

    required_paths = [
        REPO_DIR / 'CMakeLists.txt',
        REPO_DIR / 'cmake' / 'ggml.cmake',
        REPO_DIR / 'src' / 'version.cpp',
        REPO_DIR / 'thirdparty' / 'CMakeLists.txt',
        REPO_DIR / 'examples' / 'cli' / 'CMakeLists.txt',
        REPO_DIR / '.gitmodules'
    ]
    missing = [str(p) for p in required_paths if not p.is_file()]
    if missing:
        raise RuntimeError('ELLA sparse checkout is incomplete: ' + ', '.join(missing))

    # The upstream examples CMake file also configures the unused web server.
    # Replace it only in this Kaggle working copy; keep the CLI target intact.
    (REPO_DIR / 'examples' / 'CMakeLists.txt').write_text(
        'add_subdirectory(cli)\n', encoding='utf-8')

    # Fetch the pinned ggml commit as a PARTIAL CLONE, then checkout only
    # the source tree needed by the CUDA backend. A normal submodule update
    # downloads every ggml blob and is the main avoidable transfer.
    ggml_dir = REPO_DIR / 'ggml'
    if not (ggml_dir / '.git').exists():
        gitmodules = (REPO_DIR / '.gitmodules').read_text(encoding='utf-8')
        if 'https://github.com/leejet/ggml.git' not in gitmodules:
            raise RuntimeError('ELLA: unexpected ggml submodule URL; refusing unsafe checkout.')
        gitlink = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'ls-tree', 'HEAD', 'ggml'],
            check=True, capture_output=True, text=True).stdout.strip().split()
        if len(gitlink) < 3 or gitlink[1] != 'commit':
            raise RuntimeError('ELLA: pinned ggml commit was not found in the source tree.')
        ggml_sha = gitlink[2]
        if ggml_dir.exists():
            shutil.rmtree(ggml_dir)
        run(['git', 'clone', '--filter=blob:none', '--no-checkout',
             '--single-branch', 'https://github.com/leejet/ggml.git', ggml_dir])
        run(['git', '-C', ggml_dir, 'fetch', '--depth=1', '--filter=blob:none',
             'origin', ggml_sha])
        run(['git', '-C', ggml_dir, 'sparse-checkout', 'init', '--no-cone'])
        ggml_sparse = [
            '/CMakeLists.txt', '/cmake/', '/include/',
            '/src/CMakeLists.txt', '/src/ggml-version.h.in',
            '/src/ggml-cuda/', '/src/ggml-cpu/', '/src/ggml-base/'
        ]
        run(['git', '-C', ggml_dir, 'sparse-checkout', 'set', '--no-cone', *ggml_sparse])
        run(['git', '-C', ggml_dir, 'checkout', '--detach', ggml_sha])
    else:
        # Existing checkout: keep it and only ensure the required CUDA tree exists.
        if not (ggml_dir / 'CMakeLists.txt').is_file():
            raise RuntimeError('ELLA: existing ggml checkout is incomplete; delete the Kaggle working copy and rerun.')
    ggml_required = [
        ggml_dir / 'src' / 'ggml-version.h.in',
        ggml_dir / 'src' / 'ggml-cuda' / 'CMakeLists.txt',
        ggml_dir / 'src' / 'ggml-cpu' / 'CMakeLists.txt',
        ggml_dir / 'src' / 'ggml-base' / 'CMakeLists.txt'
    ]
    ggml_missing = [str(p) for p in ggml_required if not p.is_file()]
    if ggml_missing:
        raise RuntimeError('ELLA: GGML sparse checkout is missing required build files: ' + ', '.join(ggml_missing))

    # Work around Kaggle images where CMake cannot resolve CUDA::cuda_driver.
    CMAKE_FILE = REPO_DIR / 'CMakeLists.txt'
    cmake_text = CMAKE_FILE.read_text(encoding='utf-8')
    patch_marker = '# ELLA_KAGGLE_CUDA_DRIVER_TARGET'
    if patch_marker not in cmake_text:
        anchor = 'find_package(CUDAToolkit REQUIRED)'
        if anchor not in cmake_text:
            raise RuntimeError('ELLA: upstream CMake CUDA section changed; refusing unsafe patch.')
        driver_patch = r'''
# ELLA_KAGGLE_CUDA_DRIVER_TARGET
if(NOT TARGET CUDA::cuda_driver)
  find_library(ELLA_CUDA_DRIVER_LIBRARY NAMES cuda
    HINTS
      "${CUDAToolkit_LIBRARY_DIR}"
      "${CUDAToolkit_ROOT}/lib64"
      "${CUDAToolkit_ROOT}/targets/x86_64-linux/lib/stubs"
      "/usr/local/cuda/lib64/stubs"
      "/usr/local/cuda/targets/x86_64-linux/lib/stubs"
      "/usr/local/cuda/compat"
      "/usr/lib/x86_64-linux-gnu")
  if(NOT ELLA_CUDA_DRIVER_LIBRARY)
    find_file(ELLA_CUDA_DRIVER_LIBRARY NAMES libcuda.so.1
      PATHS "/usr/lib/x86_64-linux-gnu" "/usr/local/cuda/compat")
  endif()
  if(NOT ELLA_CUDA_DRIVER_LIBRARY)
    message(FATAL_ERROR "ELLA: CUDA driver library was not found.")
  endif()
  add_library(CUDA::cuda_driver UNKNOWN IMPORTED GLOBAL)
  set_target_properties(CUDA::cuda_driver PROPERTIES
    IMPORTED_LOCATION "${ELLA_CUDA_DRIVER_LIBRARY}")
endif()
'''
        CMAKE_FILE.write_text(cmake_text.replace(anchor, anchor + '\n' + driver_patch, 1),
                              encoding='utf-8')

    configure = [
        'cmake', '-S', str(REPO_DIR), '-B', str(BUILD_DIR),
        '-DCMAKE_BUILD_TYPE=Release', '-DSD_CUDA=ON',
        '-DSD_BUILD_EXAMPLES=ON', '-DSD_WEBP=OFF', '-DSD_WEBM=OFF',
        '-DSD_CUDA_CUTLASS_PATH:PATH=',
        f'-DCMAKE_CUDA_ARCHITECTURES={gpu_arch}'
    ]
    run(configure)

    mem_kb = 0
    for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemAvailable:'):
            mem_kb = int(line.split()[1])
            break
    jobs = min(os.cpu_count() or 2, 4)
    if mem_kb and mem_kb < 12 * 1024 * 1024:
        jobs = min(jobs, 2)

    print(f'ELLA: compiling only sd-cli with {jobs} jobs; build cache retained.', flush=True)
    run(['cmake', '--build', BUILD_DIR, '--config', 'Release',
         '--target', 'sd-cli', '--parallel', str(jobs)])

    if not SDCLI_PATH.is_file():
        raise RuntimeError(f'ELLA: build completed but sd-cli is missing: {SDCLI_PATH}')
    check = subprocess.run([str(SDCLI_PATH), '--help'], capture_output=True,
                           text=True, timeout=30)
    if check.returncode != 0:
        raise RuntimeError('ELLA: newly built sd-cli failed its --help smoke test: ' +
                           (check.stderr[-3000:] if check.stderr else check.stdout[-3000:]))

SDCLI = str(SDCLI_PATH)
print('ELLA CUDA CLI VERIFIED:', SDCLI, flush=True)


## 2. Download the model components (Q4)
The generator alone is not enough: Wan also requires its matching VAE and UMT5 text encoder.

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_DIR = pathlib.Path('/kaggle/working/ella_models'); MODEL_DIR.mkdir(exist_ok=True)
def get_file(repo, name):
    target = MODEL_DIR / name
    if not target.exists():
        hf_hub_download(repo_id=repo, filename=name, local_dir=str(MODEL_DIR))
    assert target.exists(), f'Missing download: {target}'
    print(name, round(target.stat().st_size/1e9, 2), 'GB')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q4_K_S.gguf')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'wan2.2_vae.safetensors')
get_file('off-grid-ai/Video-Companions-GGUF', 'umt5-xxl-encoder-Q4_K_M.gguf')


## 3. T4 validation profile

The manual test below is five seconds only as a quick smoke test. The connected worker preserves the requested scene duration and continues longer scenes in five-second chunks.

In [ ]:
PROMPT = (
    "A young woman walks briskly through a rainy Istanbul street at night. "
    "She takes clear, energetic natural steps, her arms and body move naturally, "
    "her coat and hair flow in the wind. Cars drive past in the background, "
    "rain falls visibly, water splashes under her feet, pedestrians move behind her. "
    "The camera smoothly tracks alongside her with dynamic cinematic movement. "
    "Continuous realistic motion, lively environment, highly detailed face, "
    "realistic body anatomy, cinematic lighting, photorealistic film quality."
)
NEGATIVE_PROMPT = (
    "static frame, frozen body, slow motion, barely moving, stiff movement, "
    "blurry, low quality, flicker, distorted anatomy, extra limbs, extra fingers, "
    "deformed face, duplicate people, subtitles, text, watermark."
)
MODE = 'text'
REFERENCE_IMAGE = '/kaggle/working/reference.png'

DURATION_SECONDS = 5  # manual smoke test only
FPS = 8                # source FPS; finishing can interpolate to 24 FPS
WIDTH, HEIGHT = 832, 480
STEPS = 20
SEED = 42

print('ELLA FAST MOTION PROFILE READY')
print('Duration:', DURATION_SECONDS, 'seconds')
print('FPS:', FPS)
print('Resolution:', WIDTH, 'x', HEIGHT)
print('Steps:', STEPS)
assert DURATION_SECONDS > 0 and FPS > 0 and STEPS == 20


In [ ]:
!mkdir -p /kaggle/working/ella_engine
print('ELLA runner will be fetched from the project branch in the next cell.')


## 3.5 Install AI finishing models (one time per Kaggle session)\nInstalls MuseTalk 1.5 and Real-ESRGAN into an isolated Python 3.10 environment, downloads their model weights, and configures ELLA. This cell does not generate a video.\n

In [ ]:
# ==========================================
# ELLA AI FINISHING MODELS — ONE-TIME SETUP
# MuseTalk 1.5 + Real-ESRGAN (isolated Python 3.10)
# Repairs uv environments that were created without pip.
# Does NOT generate a video.
# ==========================================
import os, pathlib, subprocess, shutil, sys
from huggingface_hub import hf_hub_download

ELLA_AI_ROOT = pathlib.Path('/kaggle/working/ella_ai_models')
MUSE_HOME = pathlib.Path('/kaggle/working/MuseTalk')
ESRGAN_HOME = pathlib.Path('/kaggle/working/Real-ESRGAN')
AI_ENV = pathlib.Path('/kaggle/working/ella_ai_env')
AI_PY = AI_ENV / 'bin' / 'python'

def run_setup(cmd, label, cwd=None):
    print('[ELLA SETUP]', label, flush=True)
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

# Install uv and locate it reliably, including Kaggle's user bin directory.
run_setup([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], 'Install isolated environment manager')
UV = shutil.which('uv')
if not UV:
    candidate = pathlib.Path.home() / '.local' / 'bin' / 'uv'
    if candidate.exists():
        UV = str(candidate)
if not UV:
    raise RuntimeError('uv installation failed; executable was not found.')

run_setup([UV, 'python', 'install', '3.10'], 'Install Python 3.10 runtime')

# uv creates unseeded environments without pip by default.
# Preserve a working environment; recreate it with --seed only if pip is absent.
pip_ok = False
if AI_PY.exists():
    check = subprocess.run(
        [str(AI_PY), '-m', 'pip', '--version'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    pip_ok = check.returncode == 0

if not pip_ok:
    if AI_ENV.exists():
        print('[ELLA SETUP] Recreating incomplete venv with pip included', flush=True)
        shutil.rmtree(AI_ENV)
    run_setup(
        [UV, 'venv', '--seed', '--python', '3.10', str(AI_ENV)],
        'Create seeded Python 3.10 environment with pip'
    )

if not AI_PY.exists():
    raise RuntimeError('Python 3.10 environment was not created.')

def ai_pip(*packages):
    run_setup(
        [str(AI_PY), '-m', 'pip', 'install', '--disable-pip-version-check', *packages],
        'Install: ' + ' '.join(packages[:3])
    )

# Verify pip before any large package installation.
run_setup([str(AI_PY), '-m', 'pip', '--version'], 'Verify pip bootstrap')

# MuseTalk's published inference stack in an isolated environment.
if not (AI_ENV / '.ella_base_installed').exists():
    ai_pip('--upgrade', 'pip', 'setuptools', 'wheel')
    ai_pip('--index-url', 'https://download.pytorch.org/whl/cu118',
           'torch==2.0.1', 'torchvision==0.15.2', 'torchaudio==2.0.2')
    ai_pip('diffusers==0.30.2', 'accelerate==0.28.0', 'numpy==1.23.5',
           'opencv-python==4.9.0.80', 'soundfile==0.12.1',
           'transformers==4.39.2', 'huggingface_hub==0.30.2',
           'librosa==0.11.0', 'einops==0.8.1', 'gdown', 'requests',
           'imageio[ffmpeg]', 'omegaconf', 'ffmpeg-python', 'moviepy',
           'PyYAML', 'tqdm', 'scipy', 'safetensors')
    ai_pip('openmim')
    run_setup([str(AI_PY), '-m', 'mim', 'install', 'mmengine'], 'Install MMEngine')
    run_setup([str(AI_PY), '-m', 'mim', 'install', 'mmcv==2.0.1'], 'Install MMCV')
    run_setup([str(AI_PY), '-m', 'mim', 'install', 'mmdet==3.1.0'], 'Install MMDetection')
    run_setup([str(AI_PY), '-m', 'mim', 'install', 'mmpose==1.1.0'], 'Install MMPose')
    (AI_ENV / '.ella_base_installed').write_text('ok')

# Clone the official inference repositories only once.
if not (MUSE_HOME / 'scripts' / 'inference.py').exists():
    if MUSE_HOME.exists():
        shutil.rmtree(MUSE_HOME)
    run_setup(['git', 'clone', '--depth', '1',
               'https://github.com/TMElyralab/MuseTalk.git', str(MUSE_HOME)],
              'Download official MuseTalk source')

if not (ESRGAN_HOME / 'inference_realesrgan.py').exists():
    if ESRGAN_HOME.exists():
        shutil.rmtree(ESRGAN_HOME)
    run_setup(['git', 'clone', '--depth', '1',
               'https://github.com/xinntao/Real-ESRGAN.git', str(ESRGAN_HOME)],
              'Download official Real-ESRGAN source')

# Real-ESRGAN inference dependencies.
ai_pip('basicsr', 'facexlib', 'gfpgan')
req = ESRGAN_HOME / 'requirements.txt'
if req.exists():
    ai_pip('-r', str(req))
run_setup([str(AI_PY), '-m', 'pip', 'install', '-e', str(ESRGAN_HOME)],
          'Install Real-ESRGAN package')

# Download MuseTalk 1.5 and inference model weights.
MODEL_ROOT = MUSE_HOME / 'models'
def hf_file(repo, filename, folder):
    folder = pathlib.Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    return pathlib.Path(hf_hub_download(
        repo_id=repo, filename=filename, local_dir=str(folder)
    ))

hf_file('TMElyralab/MuseTalk', 'musetalkV15/unet.pth', MODEL_ROOT)
hf_file('TMElyralab/MuseTalk', 'musetalkV15/musetalk.json', MODEL_ROOT)
hf_file('stabilityai/sd-vae-ft-mse', 'config.json', MODEL_ROOT / 'sd-vae')
hf_file('stabilityai/sd-vae-ft-mse', 'diffusion_pytorch_model.bin', MODEL_ROOT / 'sd-vae')
hf_file('openai/whisper-tiny', 'config.json', MODEL_ROOT / 'whisper')
hf_file('openai/whisper-tiny', 'pytorch_model.bin', MODEL_ROOT / 'whisper')
hf_file('openai/whisper-tiny', 'preprocessor_config.json', MODEL_ROOT / 'whisper')
hf_file('yzd-v/DWPose', 'dw-ll_ucoco_384.pth', MODEL_ROOT / 'dwpose')

# Face parsing models.
face_dir = MODEL_ROOT / 'face-parse-bisent'
face_dir.mkdir(parents=True, exist_ok=True)
face_weights = face_dir / '79999_iter.pth'
if not face_weights.exists():
    run_setup([str(AI_PY), '-m', 'gdown', '--id',
               '154JgKpzCPW82qINcVieuPH3fZ2e0P812', '-O', str(face_weights)],
              'Download face parsing weights')

resnet = face_dir / 'resnet18-5c106cde.pth'
if not resnet.exists():
    run_setup(['curl', '-fL',
               'https://download.pytorch.org/models/resnet18-5c106cde.pth',
               '-o', str(resnet)], 'Download ResNet weights')

# Real-ESRGAN 4x checkpoint.
weights_dir = ESRGAN_HOME / 'weights'
weights_dir.mkdir(exist_ok=True)
esrgan_weights = weights_dir / 'RealESRGAN_x4plus.pth'
if not esrgan_weights.exists():
    run_setup(['curl', '-fL',
               'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
               '-o', str(esrgan_weights)], 'Download Real-ESRGAN weights')

# Configure ELLA paths.
os.environ['ELLA_MUSETALK_HOME'] = str(MUSE_HOME)
os.environ['ELLA_MUSETALK_PYTHON'] = str(AI_PY)
os.environ['ELLA_REALESRGAN_HOME'] = str(ESRGAN_HOME)
os.environ['ELLA_REALESRGAN_PYTHON'] = str(AI_PY)

# Verify imports and required checkpoint files; no video generation.
run_setup([str(AI_PY), '-c',
           'import torch, diffusers, transformers, mmcv, mmpose, realesrgan; '
           'print("AI ENV OK | torch",torch.__version__,"| CUDA",torch.cuda.is_available())'],
          'Verify installed AI packages')

required_files = [
    MODEL_ROOT/'musetalkV15'/'unet.pth',
    MODEL_ROOT/'musetalkV15'/'musetalk.json',
    MODEL_ROOT/'sd-vae'/'diffusion_pytorch_model.bin',
    MODEL_ROOT/'whisper'/'pytorch_model.bin',
    MODEL_ROOT/'dwpose'/'dw-ll_ucoco_384.pth',
    face_weights, resnet, esrgan_weights
]
missing = [str(p) for p in required_files if not p.is_file()]
if missing:
    raise FileNotFoundError('AI model setup incomplete: ' + ', '.join(missing))

print('ELLA AI MODELS INSTALLED AND PATHS CONFIGURED')
print('MuseTalk:', MUSE_HOME)
print('Real-ESRGAN:', ESRGAN_HOME)
print('No video was generated in this setup cell.')


## 4. Generate MP4
This cell calls the ELLA runner with argument-safe subprocess execution (no shell interpolation).

In [ ]:
import importlib.util
import urllib.request
RUNNER_PATH = pathlib.Path('/kaggle/working/ella_engine/runner.py')
RUNNER_PATH.parent.mkdir(parents=True, exist_ok=True)
url='https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/runner.py'
# Always refresh the runner so Kaggle cannot reuse an older cached copy.
urllib.request.urlretrieve(url, RUNNER_PATH)
AUDIO_POST_URL='https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/audio_postprocess.py'
AUDIO_POST_PATH=RUNNER_PATH.with_name('audio_postprocess.py')
urllib.request.urlretrieve(AUDIO_POST_URL, AUDIO_POST_PATH)
for name in ("translation.py", "quality_models.py"):
    asset_url = "https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/" + name
    urllib.request.urlretrieve(asset_url, RUNNER_PATH.with_name(name))
spec=importlib.util.spec_from_file_location('ella_runner', RUNNER_PATH)
runner=importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
args=runner.parser().parse_args(['--prompt', PROMPT, '--negative-prompt', NEGATIVE_PROMPT, '--duration', str(DURATION_SECONDS), '--fps', str(FPS), '--width', str(WIDTH), '--height', str(HEIGHT), '--steps', str(STEPS), '--seed', str(SEED), '--output', '/kaggle/working/ella_output.mp4'])
args.sd_cli=pathlib.Path(SDCLI)
args.diffusion_model=MODEL_DIR/'Wan2.2-TI2V-5B-Q4_K_S.gguf'
args.vae=MODEL_DIR/'wan2.2_vae.safetensors'
args.text_encoder=MODEL_DIR/'umt5-xxl-encoder-Q4_K_M.gguf'
if MODE == 'image':
    args.image=pathlib.Path(REFERENCE_IMAGE)
result=runner.run(args)
print(json.dumps(result, ensure_ascii=False, indent=2))
from IPython.display import Video, display
display(Video('/kaggle/working/ella_output.mp4', embed=True))


## 5. Download your MP4
The final file is `/kaggle/working/ella_output.mp4`. Use the Kaggle file browser to download it. Keep the generation log (`ella_output.log`) if anything fails.

In [ ]:
!ls -lh /kaggle/working/ella_output.mp4 /kaggle/working/ella_output.log


## 6. Connect the worker to Ella2027

Run this cell after the manual MP4 test succeeds. Use the existing Kaggle Secret `ELLA_KAGGLE_WORKER_TOKEN`, with the same value as Render `KAGGLE_LTX_WORKER_TOKEN` and the ELLA provider key. The worker automatically continues scenes longer than five seconds in sequential GPU-safe chunks.

In [ ]:
import urllib.request, sys
WORKER_PATH = pathlib.Path('/kaggle/working/ella_engine/wan22_worker.py')
WORKER_PATH.parent.mkdir(parents=True, exist_ok=True)
WORKER_URL = 'https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/kaggle-wan22-worker/wan22_worker.py'
urllib.request.urlretrieve(WORKER_URL, WORKER_PATH)
AUDIO_POST_URL = 'https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/audio_postprocess.py'
AUDIO_POST_PATH = WORKER_PATH.with_name('audio_postprocess.py')
urllib.request.urlretrieve(AUDIO_POST_URL, AUDIO_POST_PATH)
for name in ("translation.py", "quality_models.py"):
    asset_url = 'https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/' + name
    urllib.request.urlretrieve(asset_url, WORKER_PATH.with_name(name))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts>=7.0.0', 'argostranslate>=1.9.0', 'PyYAML>=6.0'])

required = [
    pathlib.Path(RUNNER_PATH), pathlib.Path(SDCLI),
    pathlib.Path(MODEL_DIR) / 'Wan2.2-TI2V-5B-Q4_K_S.gguf',
    pathlib.Path(MODEL_DIR) / 'wan2.2_vae.safetensors',
    pathlib.Path(MODEL_DIR) / 'umt5-xxl-encoder-Q4_K_M.gguf',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing ELLA runtime files: ' + ', '.join(missing))

from kaggle_secrets import UserSecretsClient
ELLA_TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN')
if not ELLA_TOKEN:
    raise ValueError('Kaggle Secret ELLA_KAGGLE_WORKER_TOKEN is empty.')

sys.path.insert(0, str(WORKER_PATH.parent))
from wan22_worker import run_forever
print('[ELLA] Wan 2.2 worker starting. Requested scene lengths are split into GPU-safe continuation chunks.', flush=True)
run_forever(
    'https://ella2027.onrender.com', ELLA_TOKEN,
    pathlib.Path(RUNNER_PATH), pathlib.Path(MODEL_DIR),
    pathlib.Path(SDCLI), pathlib.Path('/kaggle/working/ella_jobs'),
    max_scene_seconds=5, idle_seconds=12
)